In [3]:
import pandas as pd
import numpy as np
import pickle
import json
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")

✅ Libraries loaded


In [4]:
# UPDATE this path to your Models folder
MODELS_DIR = r'C:/Users/user/JobGenie/Student/Models/'

try:
    with open(os.path.join(MODELS_DIR, 'salary_model.pkl'),       'rb') as f: sal_model  = pickle.load(f)
    with open(os.path.join(MODELS_DIR, 'role_model.pkl'),         'rb') as f: role_model = pickle.load(f)
    with open(os.path.join(MODELS_DIR, 'label_encoder_role.pkl'), 'rb') as f: le         = pickle.load(f)
    with open(os.path.join(MODELS_DIR, 'feature_columns.json'),   'r')  as f: meta       = json.load(f)
    print("✅ All 4 files loaded successfully")
except FileNotFoundError as e:
    print(f"❌ FILE NOT FOUND: {e}")
    print("   Make sure you ran 04_model_training_2.ipynb first.")

SALARY_FEATURES = meta['salary_features']
ROLE_FEATURES   = meta['role_features']
ALL_SKILLS      = meta['all_skills']
ROLE_CLASSES    = meta['role_classes']
TIER_MAPPING    = meta['tier_mapping']

print(f"\n📋 What was saved:")
print(f"   Salary model     : {meta['salary_model']}")
print(f"   Role model       : {meta['role_model']}")
print(f"   Salary R²        : {meta['salary_r2']}")
print(f"   Role accuracy    : {meta['role_accuracy']*100:.1f}%")
print(f"   Salary features  : {len(SALARY_FEATURES)}")
print(f"   Role features    : {len(ROLE_FEATURES)}  (8 numeric + 18 binary skill flags)")
print(f"   Role classes     : {ROLE_CLASSES}")

✅ All 4 files loaded successfully

📋 What was saved:
   Salary model     : Random Forest
   Role model       : XGBoost
   Salary R²        : 0.9232
   Role accuracy    : 95.7%
   Salary features  : 11
   Role features    : 26  (8 numeric + 18 binary skill flags)
   Role classes     : ['Backend Developer', 'Data Analyst', 'Data Scientist', 'DevOps Engineer', 'Junior Developer', 'ML Engineer', 'Software Engineer', 'Web Developer']


In [5]:
ADVANCED     = {'DSA', 'Machine Learning', 'Deep Learning', 'TensorFlow', 'PyTorch'}
INTERMEDIATE = {'Python', 'Java', 'React', 'AWS', 'Docker', 'Node.js', 'Django'}
BASIC        = set(ALL_SKILLS) - ADVANCED - INTERMEDIATE

def build_salary_input(tier, cgpa, internships, github_projects,
                       backlogs, hackathons, certifications,
                       aptitude_score, skills_list):
    skill_set = set(skills_list)
    row = {
        'college_tier_encoded': TIER_MAPPING[tier],
        'cgpa': cgpa, 'internships': internships,
        'github_projects': github_projects, 'backlogs': backlogs,
        'hackathons': hackathons, 'certifications': certifications,
        'advanced_skills':     len(skill_set & ADVANCED),
        'intermediate_skills': len(skill_set & INTERMEDIATE),
        'basic_skills':        len(skill_set & BASIC),
        'aptitude_score': aptitude_score,
    }
    return pd.DataFrame([row])[SALARY_FEATURES]

def build_role_input(tier, cgpa, internships, github_projects,
                     backlogs, hackathons, certifications,
                     aptitude_score, skills_list):
    skill_set = set(skills_list)
    row = {
        'college_tier_encoded': TIER_MAPPING[tier],
        'cgpa': cgpa, 'internships': internships,
        'github_projects': github_projects, 'backlogs': backlogs,
        'hackathons': hackathons, 'certifications': certifications,
        'aptitude_score': aptitude_score,
    }
    for skill in ALL_SKILLS:
        col = 'skill_' + skill.lower().replace('/', '_').replace('.', '_').replace(' ', '_')
        row[col] = 1 if skill in skill_set else 0
    return pd.DataFrame([row])[ROLE_FEATURES]

def predict(tier, cgpa, skills_list,
            internships=1, github_projects=2, backlogs=0,
            hackathons=1, certifications=2, aptitude_score=60.0):
    sal_df  = build_salary_input(tier, cgpa, internships, github_projects,
                                  backlogs, hackathons, certifications,
                                  aptitude_score, skills_list)
    role_df = build_role_input(tier, cgpa, internships, github_projects,
                                backlogs, hackathons, certifications,
                                aptitude_score, skills_list)
    salary     = sal_model.predict(sal_df)[0]
    role_enc   = role_model.predict(role_df)[0]
    role_name  = le.inverse_transform([role_enc])[0]
    role_probs = role_model.predict_proba(role_df)[0]
    top2_idx   = np.argsort(role_probs)[::-1][:2]
    top2       = [(le.classes_[i], round(role_probs[i]*100, 1)) for i in top2_idx]
    return round(float(salary), 2), role_name, top2

print("✅ Helper functions ready")

✅ Helper functions ready


In [6]:
salary, role, top2 = predict(
    tier            = 'Tier 1',
    cgpa            = 8.5,
    skills_list     = ['Machine Learning', 'Deep Learning', 'TensorFlow', 'Python'],
    internships     = 2,
    github_projects = 4,
    backlogs        = 0,
    hackathons      = 3,
    certifications  = 4,
    aptitude_score  = 75.0
)

print(f"Predicted Salary  : {salary:.2f} LPA")
print(f"Predicted Role    : {role}")
print(f"Top-2 predictions : {top2[0][0]} ({top2[0][1]}%)  |  {top2[1][0]} ({top2[1][1]}%)")

Predicted Salary  : 22.21 LPA
Predicted Role    : Data Scientist
Top-2 predictions : Data Scientist (99.0%)  |  Backend Developer (1.0%)


In [7]:
TEST_CASES = [
    {
        "name": "Strong ML student",
        "tier": "Tier 1", "cgpa": 9.0,
        "skills": ["Machine Learning", "Deep Learning", "TensorFlow", "PyTorch", "Python"],
        "aptitude": 85.0,
        "expect_role": ["ML Engineer", "Data Scientist"],
        "salary_min": 18, "salary_max": 35,
    },
    {
        "name": "Web developer profile",
        "tier": "Tier 2", "cgpa": 7.0,
        "skills": ["React", "JavaScript", "HTML/CSS", "Node.js"],
        "aptitude": 55.0,
        "expect_role": ["Web Developer"],
        "salary_min": 6, "salary_max": 14,
    },
    {
        "name": "Low CGPA, basic skills",
        "tier": "Tier 3", "cgpa": 5.5,
        "skills": ["Python", "SQL"],
        "aptitude": 38.0,
        "expect_role": ["Junior Developer", "Data Analyst", "Backend Developer"],
        "salary_min": 2, "salary_max": 7,
    },
    {
        "name": "DevOps profile",
        "tier": "Tier 2", "cgpa": 7.5,
        "skills": ["Docker", "AWS", "Linux", "Python", "Git"],
        "aptitude": 65.0,
        "expect_role": ["DevOps Engineer"],
        "salary_min": 10, "salary_max": 20,
    },
    {
        "name": "DSA focused student",
        "tier": "Tier 1", "cgpa": 8.5,
        "skills": ["DSA", "Python", "Java"],
        "aptitude": 78.0,
        "expect_role": ["Software Engineer", "Backend Developer"],
        "salary_min": 12, "salary_max": 25,
    },
    {
        "name": "Backlogs penalty test",
        "tier": "Tier 2", "cgpa": 6.0,
        "skills": ["Python", "SQL", "Excel"],
        "aptitude": 45.0,
        "backlogs": 3,
        "expect_role": ["Data Analyst", "Backend Developer", "Junior Developer"],
        "salary_min": 2, "salary_max": 8,
    },
]

print(f"{'#':<3} {'Student':<28} {'Role':<22} {'Salary':>8}  {'Role OK':<10} {'Salary OK'}")
print("-" * 85)

all_passed = True
for i, tc in enumerate(TEST_CASES, 1):
    salary, role, top2 = predict(
        tier=tc["tier"], cgpa=tc["cgpa"],
        skills_list=tc["skills"],
        aptitude_score=tc["aptitude"],
        backlogs=tc.get("backlogs", 0),
    )
    role_ok   = role in tc["expect_role"]
    salary_ok = tc["salary_min"] <= salary <= tc["salary_max"]
    all_passed = all_passed and role_ok and salary_ok

    r_icon = "✅" if role_ok   else "❌"
    s_icon = "✅" if salary_ok else "❌"
    print(f"{i:<3} {tc['name']:<28} {role:<22} {salary:>6.1f} LPA  {r_icon:<10} {s_icon}")

print()
if all_passed:
    print("✅ ALL 6 TESTS PASSED — models are behaving correctly")
else:
    print("❌ SOME TESTS FAILED — check the rows marked ❌ above")

#   Student                      Role                     Salary  Role OK    Salary OK
-------------------------------------------------------------------------------------
1   Strong ML student            ML Engineer              24.8 LPA  ✅          ✅
2   Web developer profile        Web Developer             9.3 LPA  ✅          ✅
3   Low CGPA, basic skills       Junior Developer          4.8 LPA  ✅          ✅
4   DevOps profile               DevOps Engineer          10.2 LPA  ✅          ✅
5   DSA focused student          Software Engineer        18.9 LPA  ✅          ✅
6   Backlogs penalty test        Junior Developer          7.2 LPA  ✅          ✅

✅ ALL 6 TESTS PASSED — models are behaving correctly


In [8]:
for i, tc in enumerate(TEST_CASES, 1):
    salary, role, top2 = predict(
        tier=tc["tier"], cgpa=tc["cgpa"],
        skills_list=tc["skills"],
        aptitude_score=tc["aptitude"],
        backlogs=tc.get("backlogs", 0),
    )
    role_ok   = role in tc["expect_role"]
    salary_ok = tc["salary_min"] <= salary <= tc["salary_max"]

    print(f"Test {i}: {tc['name']}")
    print(f"  Input  : {tc['tier']} | CGPA {tc['cgpa']} | Skills: {tc['skills']}")
    print(f"  Role   : {role}  {'✅' if role_ok else '❌ expected: ' + str(tc['expect_role'])}")
    print(f"  Salary : {salary:.1f} LPA  {'✅' if salary_ok else '❌ expected ' + str(tc['salary_min']) + '-' + str(tc['salary_max']) + ' LPA'}")
    print(f"  Top-2  : {top2[0][0]} ({top2[0][1]}%)  |  {top2[1][0]} ({top2[1][1]}%)")
    print()

Test 1: Strong ML student
  Input  : Tier 1 | CGPA 9.0 | Skills: ['Machine Learning', 'Deep Learning', 'TensorFlow', 'PyTorch', 'Python']
  Role   : ML Engineer  ✅
  Salary : 24.8 LPA  ✅
  Top-2  : ML Engineer (98.2%)  |  Data Scientist (1.7%)

Test 2: Web developer profile
  Input  : Tier 2 | CGPA 7.0 | Skills: ['React', 'JavaScript', 'HTML/CSS', 'Node.js']
  Role   : Web Developer  ✅
  Salary : 9.3 LPA  ✅
  Top-2  : Web Developer (97.4%)  |  Junior Developer (2.5%)

Test 3: Low CGPA, basic skills
  Input  : Tier 3 | CGPA 5.5 | Skills: ['Python', 'SQL']
  Role   : Junior Developer  ✅
  Salary : 4.8 LPA  ✅
  Top-2  : Junior Developer (99.9%)  |  Backend Developer (0.1%)

Test 4: DevOps profile
  Input  : Tier 2 | CGPA 7.5 | Skills: ['Docker', 'AWS', 'Linux', 'Python', 'Git']
  Role   : DevOps Engineer  ✅
  Salary : 10.2 LPA  ✅
  Top-2  : DevOps Engineer (99.8%)  |  Backend Developer (0.1%)

Test 5: DSA focused student
  Input  : Tier 1 | CGPA 8.5 | Skills: ['DSA', 'Python', 'Java']
  R

In [9]:
print("Same student, only CGPA changes:")
print(f"{'CGPA':<8} {'Predicted Salary':>18}")
print("-" * 28)

base_skills = ["Python", "SQL"]
prev_salary = 0
monotone    = True

for cgpa in [5.0, 6.0, 7.0, 8.0, 9.0, 10.0]:
    apt = round(cgpa * 7, 1)
    salary, _, _ = predict("Tier 2", cgpa, base_skills, aptitude_score=apt)
    arrow = "↑" if salary > prev_salary else ("→" if salary == prev_salary else "↓ ⚠️")
    if salary < prev_salary:
        monotone = False
    print(f"  {cgpa:<6}  {salary:>8.1f} LPA   {arrow}")
    prev_salary = salary

print()
if monotone:
    print("✅ Salary increases correctly with CGPA")
else:
    print("⚠️  Small drops exist — normal due to model noise, not a bug")

Same student, only CGPA changes:
CGPA       Predicted Salary
----------------------------
  5.0          3.5 LPA   ↑
  6.0          7.5 LPA   ↑
  7.0          9.5 LPA   ↑
  8.0         10.5 LPA   ↑
  9.0         10.4 LPA   ↓ ⚠️
  10.0        10.5 LPA   ↑

⚠️  Small drops exist — normal due to model noise, not a bug


In [10]:
print("Same student, only backlogs changes:")
print(f"{'Backlogs':<12} {'Predicted Salary':>18}")
print("-" * 32)

prev_salary = 9999
penalty_ok  = True

for bl in [0, 1, 2, 3, 4]:
    salary, _, _ = predict("Tier 2", 7.0, ["Python", "Java", "SQL"],
                           backlogs=bl, aptitude_score=52.0)
    arrow = "↓" if salary < prev_salary else ("→" if salary == prev_salary else "↑ ⚠️")
    if bl > 0 and salary > prev_salary:
        penalty_ok = False
    print(f"  {bl:<10}  {salary:>8.1f} LPA   {arrow}")
    prev_salary = salary

print()
if penalty_ok:
    print("✅ Backlogs penalty is working correctly")
else:
    print("⚠️  Salary went up with more backlogs — check your dataset generation")

Same student, only backlogs changes:
Backlogs       Predicted Salary
--------------------------------
  0                9.5 LPA   ↓
  1                9.4 LPA   ↓
  2                9.4 LPA   ↓
  3                7.7 LPA   ↓
  4                7.5 LPA   ↓

✅ Backlogs penalty is working correctly


In [11]:
print("Edge case 1: Weakest possible student")
try:
    s, r, t = predict("Tier 3", 4.0, ["Python"],
                      internships=0, github_projects=0, backlogs=5,
                      hackathons=0, certifications=0, aptitude_score=10.0)
    print(f"  Role: {r}  |  Salary: {s:.1f} LPA  ✅ No crash")
except Exception as e:
    print(f"  ❌ CRASHED: {e}")

print()
print("Edge case 2: Strongest possible student")
try:
    s, r, t = predict("Tier 1", 10.0,
                      ["Machine Learning","Deep Learning","TensorFlow","PyTorch","Python"],
                      internships=5, github_projects=10, backlogs=0,
                      hackathons=8, certifications=8, aptitude_score=100.0)
    print(f"  Role: {r}  |  Salary: {s:.1f} LPA  ✅ No crash")
except Exception as e:
    print(f"  ❌ CRASHED: {e}")

print()
print("Edge case 3: Only one skill")
try:
    for skill in ["DSA", "React", "Docker", "Excel"]:
        s, r, t = predict("Tier 2", 7.0, [skill], aptitude_score=52.0)
        print(f"  Only '{skill}'  →  {r}  ({s:.1f} LPA)  ✅")
except Exception as e:
    print(f"  ❌ CRASHED: {e}")

Edge case 1: Weakest possible student
  Role: Junior Developer  |  Salary: 3.5 LPA  ✅ No crash

Edge case 2: Strongest possible student
  Role: ML Engineer  |  Salary: 27.5 LPA  ✅ No crash

Edge case 3: Only one skill
  Only 'DSA'  →  Software Engineer  (14.4 LPA)  ✅
  Only 'React'  →  Software Engineer  (9.4 LPA)  ✅
  Only 'Docker'  →  Backend Developer  (9.4 LPA)  ✅
  Only 'Excel'  →  Data Analyst  (6.6 LPA)  ✅


In [12]:
# This is what the frontend form will POST to Flask
sample_api_input = {
    "college_tier":    "Tier 1",
    "cgpa":            8.5,
    "internships":     2,
    "github_projects": 4,
    "backlogs":        0,
    "hackathons":      3,
    "certifications":  4,
    "aptitude_score":  75.0,
    "skills":          ["Machine Learning", "Deep Learning", "TensorFlow", "Python"]
}

# Simulate what Flask will do internally
salary, role, top2 = predict(
    tier             = sample_api_input["college_tier"],
    cgpa             = sample_api_input["cgpa"],
    skills_list      = sample_api_input["skills"],
    internships      = sample_api_input["internships"],
    github_projects  = sample_api_input["github_projects"],
    backlogs         = sample_api_input["backlogs"],
    hackathons       = sample_api_input["hackathons"],
    certifications   = sample_api_input["certifications"],
    aptitude_score   = sample_api_input["aptitude_score"],
)

# This is what Flask will return as JSON
api_response = {
    "predicted_salary_lpa": salary,
    "predicted_role":        role,
    "top_roles": [
        {"role": top2[0][0], "confidence": top2[0][1]},
        {"role": top2[1][0], "confidence": top2[1][1]},
    ]
}

import json as _json
print("📥 Frontend sends this JSON:")
print(_json.dumps(sample_api_input, indent=2))
print()
print("📤 Flask returns this JSON:")
print(_json.dumps(api_response, indent=2))

📥 Frontend sends this JSON:
{
  "college_tier": "Tier 1",
  "cgpa": 8.5,
  "internships": 2,
  "github_projects": 4,
  "backlogs": 0,
  "hackathons": 3,
  "certifications": 4,
  "aptitude_score": 75.0,
  "skills": [
    "Machine Learning",
    "Deep Learning",
    "TensorFlow",
    "Python"
  ]
}

📤 Flask returns this JSON:
{
  "predicted_salary_lpa": 22.21,
  "predicted_role": "Data Scientist",
  "top_roles": [
    {
      "role": "Data Scientist",
      "confidence": 99.0
    },
    {
      "role": "Backend Developer",
      "confidence": 1.0
    }
  ]
}


In [13]:
print("=" * 55)
print("  TESTING SUMMARY")
print("=" * 55)
print(f"  Salary model  : {meta['salary_model']}")
print(f"  Role model    : {meta['role_model']}")
print(f"  Salary R²     : {meta['salary_r2']}  (training metric)")
print(f"  Role accuracy : {meta['role_accuracy']*100:.1f}%  (training metric)")
print()
print("  Tests run:")
print("    ✓ 6 student profile test cases")
print("    ✓ Monotonicity check (CGPA vs salary)")
print("    ✓ Backlogs penalty check")
print("    ✓ Edge cases (min/max/single-skill)")
print("    ✓ Flask API input/output preview")
print()
print("  If all checks passed → proceed to Step 8: Flask API")
print("=" * 55)

  TESTING SUMMARY
  Salary model  : Random Forest
  Role model    : XGBoost
  Salary R²     : 0.9232  (training metric)
  Role accuracy : 95.7%  (training metric)

  Tests run:
    ✓ 6 student profile test cases
    ✓ Monotonicity check (CGPA vs salary)
    ✓ Backlogs penalty check
    ✓ Edge cases (min/max/single-skill)
    ✓ Flask API input/output preview

  If all checks passed → proceed to Step 8: Flask API
